## LLMOps - Langfuse 설치

## 디버깅의 어려움
- 전통적인 소프트웨어는 스택 트레이스는 로그를 통해 문제를 추적할 수 있음.
- 하지만 LLM 어플리케이션에서는 다음과 같은 어려움 존재

### 디버깅의 어려움  
**문제상황**   
- 사용자가 "답변이 이상해요"라고 보고
- 어떤 프롬프트가 사용되었는지 알 수 없음.
- 모델이 어떤 컨텍스트를 받았는지 불명확
- 중간 단계의 에이전트 동작을 추적할 수 없음.

**필요한 정보**
- 전체 프롬프트(시스템메시지+사용자입력+컨텍스트)
- 모델 파라미터
- 응답생성시간
- 토큰 사용량 및 비용
- 에이전트의 도구 호출 순서

## 비용관리의 복잡성
- LLM API 호출은 토큰 단위로 과금되며, 비용을 예측하기 어려움

## 품질 평가의 주관성
- 전통적인 소프트웨어는 단위 테스트로 검증이 가능함.
- LLM의 출력은 정답이 주관적이며, 명확하게 답이 정해져있지 않음.

## 프로덕션 모니터링의 필요성
- 개발환경에서 잘 작동하던 LLM 애플리케이션이 프로덕션에서 문제를 일으킬 수 있음.   

**모니터링 해야 할 지표**
- 응답 품질(사용자 피드백, 평가 점수)
- 응답시간(레이턴시)
- 비용(일일/월간 토큰 사용량)
- 에러율(API실패, 타임아웃)
- 사용패턴(어떤 기능이 많이 사용되는가)

## LLMOps의 등장
- 이러한 도전과제들을 해결하기 위해 LLMOps(Large Language Model Operations)라는 새로운 분야 등장

**LLMOps의 핵심 요소**
- 관찰성(Observability): 모든 LLM 호출을 추적하고 로깅
- 평가(Evaluation): 자동화된 품질 평가 프레임워크
- 모니터링(Monitoring): 실시간 성능 및 비용 대시보드
- 실험 관리(Experimentation): 프롬프트 버전 관리 및 A/B 테스트
- 데이터관리(Data Management): 사용자 입력/출력 데이터 수집 및 분석

```{markdown}
MLOps 와의 차이점
MLOps는 전통적인 머신러닝 모델의 학습/배포/모니터링에 집중합니다. LLMOps는 이미 학습된 대형 모델을 프롬프트와 컨텍스트로 제어하는 데 초점을 맞춥니다. 따라서 프롬프트 엔지니어링, 토큰 비용 관리, 에이전트 추적과 같은 고유한 도구가 필요합니다.
```

## Langfuse 가 해결하는 문제
- Langfuse는 위에서 언급한 도전 과제를 해결하기 위한 오픈소스 LLMOps 플랫폼임.

### Langfuse의 핵심 기능
- 트레이싱: 모든 LLM 호출과 에이전트 동작을 자동으로 기록
- 비용추적: 토큰 사용량과 비용을 실시간으로 집계
- 프롬프트 관리: 프롬프트를 버전별로 관리하고 배포
- 평가: 자동화된 평가 파이프라인 구축
- 대쉬보드: 사용 패턴과 성능 지표를 UI로 시각화

## Langfuse 핵심 개념
- Langfuse의 데이터모델은 트레이스(Trace)와 관찰(Observation)으로 구성됨.

### Trace
- 하나의 사용자 요청에 대한 전체 실행 흐름

### 관찰(Observation)
- 트레이스 내부의 개별 작업 단위. 3가지 타입으로 구성됨

1. Span: 일반적인 작업(예: 데이터베이스 쿼리, 함수 실행 등등)
2. Generation: LLM 호출(입력, 출력, 모델, 토큰 수 포함)
3. Event: 특정 시점의 이벤트(로그, 메트릭 등)


In [ ]:
from langfuse.openai import openai
from langfuse import observe

@observe()  # 이 함수는 하나의 Span이 됨
def retrieve_documents(query):
    # 문서 검색 로직
    return documents

@observe()  # 이 함수도 Span이 됨
def generate_answer(documents, query):
    # LLM 호출 → Generation으로 자동 기록
    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": f"Documents: {documents}\n\nQuestion: {query}"}
        ]
    )
    return response.choices[0].message.content

# 전체 실행 흐름
@observe()  # 최상위 트레이스
def rag_pipeline(user_query):
    docs = retrieve_documents(user_query)  # Span
    answer = generate_answer(docs, user_query)  # Span + Generation
    return answer


### 계층 구조
```
Trace: rag_pipeline
├── Span: retrieve_documents
│   └── Event: found 5 documents
└── Span: generate_answer
    └── Generation: gpt-4o call
        ├── Input tokens: 150
        ├── Output tokens: 80
        └── Cost: $0.0032
```

### Langfuse 아키텍쳐 구성
1. langfuse SDK
2. Langfuse server: 트레이스 데이터를 저장하고 대쉬보드를 제공하는 백엔드 서버   
   - API 엔드포인트: SDK에서 트레이스를 전송받음
   - 데이터베이스: PostgreSQL에 트레이스 저장
   - 웹대쉬보드: React 기반 UI 제공

3. Langfuse UI
- 웹 브라우저에서 접근 가능한 대쉬보드  
   
  - 트레이스 목록 및 상세 보기
  - 비용 및 사용량 분석
  - 프롬프트 관리
  - 평가 결과 확인

## Langfuse가 제공하는 가치
### 1. 개발 단계

- 프롬프트 실험 결과를 즉시 확인
- 에이전트 동작을 시각적으로 디버깅
- 로컬 개발 시 트레이스 확인 가능

### 2. 스테이징 단계

- 통합 테스트 결과를 트레이스로 기록
- 성능 병목 지점 파악
- 비용 추정

### 3. 프로덕션 단계

- 실시간 모니터링 대시보드
- 사용자 피드백과 트레이스 연결
- 이상 감지 및 알림
- 비용 초과 방지

## Langfuse Cloud VS 셀프 호스팅

### Langfuse Cloud
- 설치 불필요
- 즉시 사용가능
- EU/US 리전 선택 가능
- 프로토타입이나 중소규모 프로젝트에 적합

### 셀프 호스팅
- 데이터를 자체 서버에 보관
- 커스터마이징 가능
- 규제 요구사항이 있거나 대규모 트래픽을 처리하는 기업에 적합

## Setting

### Hello langfuse

In [ ]:
from langfuse.openai import openai
from dotenv import load_dotenv

load_dotenv()
response = openai.chat.completions.create(
    name = "hello-langfuse",
    model = "gpt-4o-mini",
    messages=[
        {"role": "system" , "content": "you are a helpful assistant"},
        {"role": "user" , "content": "hello langfuse"}
    ],
    metadata = {"environment": "development"}
)

print(response.choices[0].message.content)

### 함수 데코레이터 사용
- python에서는 ```@observe``` 데코레이터를 사용하여 모든 함수를 자동으로 트레이스 할 수 있음.